In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =============================================================
# bilingual topic modeling: EN/ZH chromium corpus
# Complements the framing classifier — shows what kind of topics dominate
# discourse in each language, not just security/economic framing.
# =============================================================
# structure:
#   /content/drive/MyDrive/chromium_project/pdfs/en/*.pdf
#   /content/drive/MyDrive/chromium_project/pdfs/zh/*.pdf
#   /content/drive/MyDrive/chromium_project/data/  <- outputs go there

!pip install -q --force-reinstall "pillow==10.4.0"
!pip install -q transformers pdfplumber torch bertopic sentence-transformers umap-learn hdbscan

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import pdfplumber
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

BASE = "/content/drive/MyDrive/chromium_project"
PDF_EN_DIR = f"{BASE}/pdfs/en"
PDF_ZH_DIR = f"{BASE}/pdfs/zh"
OUT_DIR = f"{BASE}/data"
os.makedirs(OUT_DIR, exist_ok=True)

# Text extraction + chunking
def extract_text(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

def split_paragraphs(text, min_len=40, chunk_size=800):
    text = re.sub(r'\s+', ' ', text).strip()
    sentences = re.split(r'(?<=[.!?。！？])\s*', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    chunks = []
    current = ""
    for sent in sentences:
        if len(current) + len(sent) <= chunk_size:
            current += (" " if current else "") + sent
        else:
            if len(current) >= min_len:
                chunks.append(current)
            current = sent
    if len(current) >= min_len:
        chunks.append(current)
    return chunks

# buiding the CORPUS part
# NOTE: this uses ALL chunks, not just chromium-keyword-filtered ones like the
# framing pipeline. The point here is to see what topics dominate each
# language's discourse overall - filtering to chromium-only chunks would
# just re-derive a narrower version of the framing analysis.
def build_corpus():
    records = []
    for lang, folder in [("en", PDF_EN_DIR), ("zh", PDF_ZH_DIR)]:
        if not os.path.exists(folder):
            print(f"Folder not found: {folder}")
            continue
        for fname in os.listdir(folder):
            if not fname.lower().endswith(".pdf"):
                continue
            path = os.path.join(folder, fname)
            print(f"Extracting {fname} ({lang})...")
            text = extract_text(path)
            chunks = split_paragraphs(text)
            for chunk in chunks:
                records.append({"file": fname, "lang": lang, "text": chunk})
    return pd.DataFrame(records)

corpus_df = build_corpus()
print(f"\nTotal chunks: {len(corpus_df)} ({(corpus_df['lang']=='en').sum()} en, {(corpus_df['lang']=='zh').sum()} zh)")

# multilingual embeddings
# paraphrase-multilingual-mpnet-base-v2 embeds EN and ZH into the SAME vector
# space, which is what lets BERTopic find topics that span both languages
# rather than producing separate, non-comparable topic sets per language.
embed_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
embeddings = embed_model.encode(corpus_df["text"].tolist(), show_progress_bar=True)

# BERTopic
topic_model = BERTopic(
    embedding_model=embed_model,
    min_topic_size=8,
    calculate_probabilities=False,
    verbose=True
)

topics, _ = topic_model.fit_transform(corpus_df["text"].tolist(), embeddings)
corpus_df["topic"] = topics

# inspect topics
topic_info = topic_model.get_topic_info()
print("\n=== Topic overview ===")
print(topic_info)

# -1 is BERTopic's outlier bucket (chunks that didn't fit any topic well).
# A large -1 count on a small corpus is common — note this as a limitation
# rather than treating every topic assignment as equally confident.
n_outliers = (corpus_df["topic"] == -1).sum()
print(f"\n{n_outliers}/{len(corpus_df)} chunks assigned to outlier topic (-1)")

# topic distribution
# This is the key comparison for your framing RQ: do EN and ZH sources
# cluster around different topics, or the same topics in different proportions?
topic_by_lang = pd.crosstab(corpus_df["topic"], corpus_df["lang"], normalize="columns")
topic_by_lang = topic_by_lang.merge(
    topic_info[["Topic", "Name"]], left_index=True, right_on="Topic"
).set_index("Name")[["en", "zh"]]

print("\n=== Topic share by language (excluding outlier row -1 for readability) ===")
print(topic_by_lang.drop(index=[c for c in topic_by_lang.index if "-1" in str(c)], errors="ignore"))

# store outputs somewhere!
corpus_df.to_csv(f"{OUT_DIR}/topic_modeling_chunks.csv", index=False)
topic_info.to_csv(f"{OUT_DIR}/topic_modeling_topics.csv", index=False)
topic_by_lang.to_csv(f"{OUT_DIR}/topic_by_language.csv")
print(f"\nSaved: topic_modeling_chunks.csv, topic_modeling_topics.csv, topic_by_language.csv in {OUT_DIR}")

# visualization (not sure if this part works properly)
fig = topic_model.visualize_barchart(top_n_topics=10)
fig.show()

fig2 = topic_model.visualize_topics()
fig2.show()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdfplumber 0.11.10 requires Pillow>=12.2.0, but you have pillow 10.4.0 which is incompatible.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting 1-s2.0-S2950555024000120-main.pdf (en)...
Extracting Let’s Get Critical Critical Minerals Mini Deals as Evolving.pdf (en)...
Extracting Allies and enemies On the political determinants of trade in critical minerals.pdf (en)...
Extracting To What Extend Complex Interdependence Theorists Challenge to Structural Realist.pdf (en)...
Extracting The Geopolitics of Mineral Resources for Renewable Energy Technologies.pdf (en)...
Extracting The Dispossessed Large-Scale Land Acquisitions,.pdf (en)...
Extracting FROM BALANCING TO ACTIVE HEDGING.pdf (en)...
Extracting Kazakhstan After the Double 

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

2026-08-11 15:08:52,288 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-11 15:09:06,798 - BERTopic - Dimensionality - Completed ✓
2026-08-11 15:09:06,801 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-11 15:09:06,855 - BERTopic - Cluster - Completed ✓
2026-08-11 15:09:06,866 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-11 15:09:07,147 - BERTopic - Representation - Completed ✓



=== Topic overview ===
    Topic  Count                                              Name  \
0      -1    298                               -1_the_and_ｈｔｔｐｓ_of   
1       0    163                           0_resource_cmtis_the_of   
2       1     83               1_trade_political_dissimilarity_the   
3       2     72                       2_of_the_international_that   
4       3     65                          3_pre_treatment_the_deal   
5       4     54                       4_land_and_rights_customary   
6       5     51                      5_kazakhstan_projects_and_to   
7       6     47               6_chromium_china_chromite_resources   
8       7     47                            7_kazakhstan_the_of_in   
9       8     38                              8_and_hedging_the_of   
10      9     34                         9_mineral_energy_the_will   
11     10     33                           10_kazakhstan_in_the_to   
12     11     33                      11_www_canada_https_critical

In [1]:
!git config --global user.email "pakmv2006@gmail.com"

In [2]:
!git config --global user.name "melomilk"

In [9]:
!find /content/drive/MyDrive -iname "topic_modeling.ipynb"

/content/drive/MyDrive/Colab Notebooks/topic_modeling.ipynb


In [11]:
!git clone https://github.com/melomilk/centralasia-china-research.git
!cp /content/drive/MyDrive/ColabNotebooks/topic_modeling.ipynb centralasia-china-research/
%cd centralasia-china-research

Cloning into 'centralasia-china-research'...
cp: cannot stat '/content/drive/MyDrive/ColabNotebooks/topic_modeling.ipynb': No such file or directory
/content/centralasia-china-research/centralasia-china-research/centralasia-china-research/centralasia-china-research/centralasia-china-research/centralasia-china-research
